# CrowdGuard — Complete Kaggle Run

This notebook contains the full setup, environment checks, smoke tests, pipeline smoke test, and the 20-client CrowdGuard experiment.

**Run the cells in order. Do not use Run All after the full experiment has started.**


## 0. Kaggle requirements

- GPU enabled
- Internet enabled
- Python 3.10 environment created with `uv`
- OpenFL 1.9
- Metaflow 2.7.15
- PyTorch with CUDA


In [1]:
# Basic Kaggle/GPU check
import sys, os
print("Notebook Python:", sys.version)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!free -h


Notebook Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Tesla T4, 15360 MiB, 0 MiB
Tesla T4, 15360 MiB, 0 MiB
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.3Gi        25Gi       1.2Mi       5.1Gi        30Gi
Swap:             0B          0B          0B


In [2]:
# Make sure uv is available
!command -v uv || python -m pip install -q uv
!uv --version


/usr/local/bin/uv
uv 0.12.9 (x86_64-unknown-linux-gnu)


In [3]:
# Clone the exact CrowdGuard branch
!rm -rf /kaggle/working/Verifiable-Federated-Unlearning
!git clone -b crowdguard-branch https://github.com/clearlynew/Verifiable-Federated-Unlearning.git /kaggle/working/Verifiable-Federated-Unlearning
!git -C /kaggle/working/Verifiable-Federated-Unlearning branch --show-current


Cloning into '/kaggle/working/Verifiable-Federated-Unlearning'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 41 (delta 7), reused 40 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 7.19 MiB | 28.63 MiB/s, done.
Resolving deltas: 100% (7/7), done.
crowdguard-branch


In [4]:
# Enter the CrowdGuard project
%cd /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
!pwd
!ls -lah


/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
total 308K
drwxr-xr-x 7 root root 4.0K Oct  3 07:46 .
drwxr-xr-x 6 root root 4.0K Oct  3 07:46 ..
-rw-r--r-- 1 root root 5.8K Oct  3 07:46 adaptive_backdoor.py
drwxr-xr-x 2 root root 4.0K Oct  3 07:46 Amsterdam
drwxr-xr-x 2 root root 4.0K Oct  3 07:46 Bangalore
drwxr-xr-x 2 root root 4.0K Oct  3 07:46 Chandler
-rw-r--r-- 1 root root  22K Oct  3 07:46 cifar10_crowdguard.py
-rw-r--r-- 1 root root  22K Oct  3 07:46 CrowdGuardClientValidation.py
drwxr-xr-x 2 root root 4.0K Oct  3 07:46 Detroit
drwxr-xr-x 2 root root 4.0K Oct  3 07:46 director
-rw-r--r-- 1 root root 2.8K Oct  3 07:46 experiment_config.py
-rw-r--r-- 1 root root  55K Oct  3 07:46 FederatedCrowdGuard.ipynb
-rw-r--r-- 1 root root   16 Oct  3 07:46 .gitignore
-rw-r--r-- 1 root root 4.6K Oct  3 07:46 lightweight_resnet18.py
-rw-r--r-- 1 root root 6.7K Oct  3 07:46 local_pipeline_smoke.py
-rw-r--r-- 1 root root 4.3

## 1. Create the Python 3.10 environment

Metaflow 2.7.15 requires the Python 3.10 environment used by the project; the Kaggle notebook's default Python 3.12 environment is not used for the experiment.

In [5]:
# Install/find Python 3.10 through uv, then create the environment
!uv python install 3.10
!rm -rf /kaggle/working/crowdguard-env
!uv venv /kaggle/working/crowdguard-env --python 3.10
!/kaggle/working/crowdguard-env/bin/python --version

Installed Python 3.10.21 in 1.35s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: /kaggle/working/crowdguard-env
Activate with: source /kaggle/working/crowdguard-env/bin/activate
Python 3.10.21


In [6]:
# Install the core versions required by the project
!uv pip install --python /kaggle/working/crowdguard-env/bin/python \
    "openfl==1.9" \
    "metaflow==2.7.15" \
    "setuptools<81"


Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 56 packages in 519ms
Prepared 56 packages in 1.39s
░░░░░░░░░░░░░░░░░░░░ [0/56] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 56 packages in 682ms
 + astroid==4.3.3
 + blinker==1.9.0
 + boto3==1.43.108
 + botocore==1.43.108
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cryptography==50.0.2
 + dill==0.4.1
 + dynaconf==3.3.5
 + flask==3.1.1
 + flatten-json==0.1.14
 + grpcio==1.65.5
 + idna==3.20
 + isort==9.0.2
 + itsdangerous==2.2.0
 + jinja2==3.1.6
 + jmespath==1.1.0
 + joblib==1.6.0
 + markdown-it-py==4.2.0
 + mark

In [7]:
# Install the remaining project dependencies
!uv pip install --python /kaggle/working/crowdguard-env/bin/python \
    torch torchvision numpy scipy scikit-learn pandas matplotlib \
    ray dill tabulate nbformat nbdev


Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 104 packages in 400ms
Prepared 76 packages in 38.50s
░░░░░░░░░░░░░░░░░░░░ [0/76] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 76 packages in 21.70s
 + anyio==4.15.1
 + asttokens==3.0.2
 + astunparse==1.6.3
 + attrs==26.1.0
 + build==1.6.1
 + contourpy==1.3.2
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + decorator==5.3.1
 + exceptiongroup==1.3.1
 + execnb==0.3.3
 + executing==2.2.1
 + fastcore==2.2.32
 + fastgit==0.1.4
 + fastjsonschema==2.22.2
 + fastspec==0.2.5
 + fasttransport==0.0.4
 + filelock==4.0.9
 + fonttools==4.65

In [8]:
# Verify the exact environment and CUDA
import subprocess

python = "/kaggle/working/crowdguard-env/bin/python"
check = r'''
import sys, torch, openfl, metaflow, numpy, scipy, sklearn, pandas
print("Python:", sys.version.split()[0])
print("OpenFL:", openfl.__version__)
print("Metaflow:", metaflow.__version__)
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
'''
subprocess.run([python, "-c", check], check=True)
!nvidia-smi


/kaggle/working/crowdguard-env/lib/python3.10/site-packages/metaflow/metaflow_config.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Python: 3.10.21
OpenFL: 1.9
Metaflow: 2.7.15
PyTorch: 2.14.1+cu130
NumPy: 2.2.6
SciPy: 1.15.3
sklearn: 1.7.2
pandas: 2.3.3
CUDA available: True
GPU: Tesla T4
Sat Oct  3 07:48:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      

In [9]:
!uv pip install \
    --python /kaggle/working/crowdguard-env/bin/python \
    --force-reinstall \
    "scikit-learn==1.6.1"

Using Python 3.10.21 environment at: /kaggle/working/crowdguard-env
Resolved 6 packages in 113ms
Prepared 6 packages in 160ms
Uninstalled 6 packages in 124ms
░░░░░░░░░░░░░░░░░░░░ [0/6] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 6 packages in 285ms
 ~ cloudpickle==3.1.2
 ~ joblib==1.6.0
 ~ numpy==2.2.6
 - scikit-learn==1.7.2
 + scikit-learn==1.6.1
 ~ scipy==1.15.3
 ~ threadpoolctl==3.7.0


In [10]:
# Metaflow needs a user identity in the Kaggle environment
import os
os.environ["USERNAME"] = "kaggle"
os.environ["USER"] = "kaggle"
print("USERNAME =", os.environ["USERNAME"])
print("USER =", os.environ["USER"])


USERNAME = kaggle
USER = kaggle


## 2. Smoke test

This checks the lightweight model, trigger generation, benign/malicious update path, CrowdGuard validation components, and historical-state handoff without launching the 20-client full workflow.

In [11]:
# Local unit/integration smoke test
!/kaggle/working/crowdguard-env/bin/python local_smoke_test.py


[1/6] Model parameters: 275,730
[2/6] Forward + internal states: OK (10 states)
[3/6] Random trigger: OK at (18, 1), target=6
[4/6] Benign + adaptive malicious local updates: OK
Distance: cosine, use y 2: [np.float64(4.162379190979621), np.float64(0.4298525020112103)]
Distance: cosine, use x 2: [np.float64(18.97044194670766), np.float64(0.4298525020112103)]
Distance: cosine, use y 1: [np.float64(12.24744871391589)]
Distance: cosine, use x 2: [np.float64(12.247448713915887), np.float64(0.0)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: euclid, use y 2: [np.float64(9.601610557185174), np.float64(0.15209944486946503)]
Distance: euclid, use x 2: [np.float64(14.323509400183331), np.float64(0.15209944486946503)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar di

## 3. Local pipeline smoke test

This exercises the workflow-level pipeline before the expensive 20-client run.

In [12]:
# Workflow/pipeline smoke test
!/kaggle/working/crowdguard-env/bin/python local_pipeline_smoke.py


Distance: cosine, use y 2: [np.float64(11.505199808549097), np.float64(0.8410280436741007)]
Distance: cosine, use x 2: [np.float64(0.8410280436741007), np.float64(9.799024545727006)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: cosine, use y 1: [np.float64(12.247448713915894)]
Distance: cosine, use x 2: [np.float64(12.247448713915894), np.float64(0.0)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W = numer / denom
Distance: euclid, use y 2: [np.float64(16.34195069112325), np.float64(3.254528784820835)]
Distance: euclid, use x 2: [np.float64(3.254528784820835), np.float64(3.407647170625277)]
/kaggle/working/crowdguard-env/lib/python3.10/site-packages/scipy/stats/_morestats.py:3057: RuntimeWarning: divide by zero encountered in scalar divide
  W 

In [13]:
# Final pre-run resource check
!free -h
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv


               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.4Gi       916Mi       1.2Mi        29Gi        29Gi
Swap:             0B          0B          0B
name, memory.total [MiB], memory.used [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 MiB, 0 %
Tesla T4, 15360 MiB, 0 MiB, 0 %


## 4. FULL 20-CLIENT CROWDGUARD EXPERIMENT

**This is the long-running cell.**

The configured experiment uses 20 clients and 10 FL rounds, with 10 local epochs per client. On the Kaggle T4 setup this previously took roughly 3 hours. Once this cell starts, **do not rerun it and do not use Run All**.

The output is also written to `crowdguard_run.log` so that the run has a persistent text record in `/kaggle/working`.

In [14]:
# FULL RUN — do not start a second copy while this is running.
!/kaggle/working/crowdguard-env/bin/python -u cifar10_crowdguard.py 2>&1 | tee crowdguard_run.log


/kaggle/working/crowdguard-env/lib/python3.10/site-packages/metaflow/metaflow_config.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Aggregator step "start" registered
Collaborator step "train" registered
Aggregator step "collect_models" registered
Collaborator step "local_validation" registered
Aggregator step "defend" registered
Aggregator step "end" registered
100%|██████████| 170M/170M [00:07<00:00, 21.6MB/s]
Creating local datastore in current directory (/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/.metaflow)

Calling start
############################################################
CrowdGuard project run
Clients: 20
Malicious clients: ['malicious_00']
Model parameters: 275,730
Trigger: 6x6, position=(18,1), target=6
CrowdGuard mode: DETECTION ONL

## 5. Inspect results AFTER the full run finishes


In [15]:
!cd /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard && \
tar -czf /kaggle/working/crowdguard_completed_run.tar.gz \
outputs .metaflow crowdguard_run.log

In [16]:
# Do not run this until the full experiment has finished.
!ls -lah /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard
!ls -lah /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/.metaflow 2>/dev/null || true
!tail -100 /kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/crowdguard_run.log 2>/dev/null || true


total 992K
drwxr-xr-x 12 root root 4.0K Oct  3 11:58 .
drwxr-xr-x  6 root root 4.0K Oct  3 07:46 ..
-rw-r--r--  1 root root 5.8K Oct  3 07:46 adaptive_backdoor.py
drwxr-xr-x  2 root root 4.0K Oct  3 07:46 Amsterdam
drwxr-xr-x  2 root root 4.0K Oct  3 07:46 Bangalore
drwxr-xr-x  2 root root 4.0K Oct  3 07:46 Chandler
-rw-r--r--  1 root root  22K Oct  3 07:46 cifar10_crowdguard.py
-rw-r--r--  1 root root  22K Oct  3 07:46 CrowdGuardClientValidation.py
-rw-r--r--  1 root root 658K Oct  3 11:58 crowdguard_run.log
drwxr-xr-x  3 root root 4.0K Oct  3 07:49 data
drwxr-xr-x  2 root root 4.0K Oct  3 07:46 Detroit
drwxr-xr-x  2 root root 4.0K Oct  3 07:46 director
-rw-r--r--  1 root root 2.8K Oct  3 07:46 experiment_config.py
-rw-r--r--  1 root root  55K Oct  3 07:46 FederatedCrowdGuard.ipynb
-rw-r--r--  1 root root   16 Oct  3 07:46 .gitignore
-rw-r--r--  1 root root 4.6K Oct  3 07:46 lightweight_resnet18.py
-rw-r--r--  1 root root 6.7K Oct  3 07:46 local_pipeline_smoke.py
-rw-r--r--  1 root ro

In [17]:
!find /kaggle/working -name "crowdguard_handoff.pt" -o -name "crowdguard_run.log"

/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/crowdguard_run.log
/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/outputs/crowdguard_handoff.pt
/kaggle/working/Verifiable-Federated-Unlearning/CrowdGuard/smoke_outputs/crowdguard_handoff.pt


## 6. Important: preserve the run

After the full run completes, use **Save Version** so the notebook/output is preserved. Before starting Wu, inspect the generated model/history/metrics files and confirm that the per-client, per-round historical updates needed for Wu are available.